# VGAE vs R-GCN Latent Space Demo

This notebook compares trained VGAE latents against the standard R-GCN embeddings on the `tkgl-smallpedia` Wikidata graph. It includes label inspection, a true Wikidata-human node subset, static latent diagnostics, and a draggable latent-space link-prediction demo.

In [ ]:
from pathlib import Path
import json
import pickle
import random
import sys
import time
import urllib.parse
import urllib.request

import numpy as np
import torch
from IPython.display import HTML, display
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import plotly.express as px


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "R-GCN" / "rgcn_model.py").exists():
            return candidate
    raise RuntimeError("Could not find project root containing R-GCN/rgcn_model.py")


PROJECT_ROOT = find_project_root(Path.cwd())
RGCN_DIR = PROJECT_ROOT / "R-GCN"
CHECKPOINT_DIR = RGCN_DIR / "checkpoints"
sys.path.insert(0, str(RGCN_DIR))

from rgcn_model import RGCNLinkPredictor
from vgae_model import VGAE

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

# Compatibility for graph checkpoints pickled from notebooks.
RelationalGraph = type("RelationalGraph", (), {})


class GraphCheckpointUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__" and name == "RelationalGraph":
            return RelationalGraph
        return super().find_class(module, name)


GRAPH_CKPT = CHECKPOINT_DIR / "wiki_graph.pkl"
RGCN_CKPT = CHECKPOINT_DIR / "rgcn_scratch_wikidata.pt"
VGAE_RESULTS = CHECKPOINT_DIR / "vgae_wikidata_results.pkl"
NODE_ID_MAP = RGCN_DIR / "datasets" / "tkgl_smallpedia" / "ml_tkgl-smallpedia_nodeid.pkl"
HUMAN_NODE_CACHE = CHECKPOINT_DIR / "wikidata_human_nodes.pkl"

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
with GRAPH_CKPT.open("rb") as f:
    graph = GraphCheckpointUnpickler(f).load()

forward_rel_count = max(graph.rel_labels) + 1
assert graph.num_relations == 2 * forward_rel_count


def rel_label(rel_id: int) -> str:
    if rel_id in graph.rel_labels:
        return graph.rel_labels[rel_id]
    reverse_id = rel_id - forward_rel_count
    if reverse_id in graph.rel_labels:
        return "reverse: " + graph.rel_labels[reverse_id]
    return str(rel_id)


print(f"Graph: {graph.name}")
print(f"Nodes: {graph.num_nodes:,}; node labels: {len(graph.node_labels):,}")
print(f"Relations: {graph.num_relations:,}; forward relation labels: {len(graph.rel_labels):,}")
print("\nRelation label examples:")
for rel_id, label in list(graph.rel_labels.items())[:20]:
    print(f"  {rel_id:3d}: {label}")

print("\nNode label examples:")
for node_id, label in list(graph.node_labels.items())[:20]:
    print(f"  {node_id:5d}: {label}")

## True Human Node Subset

The processed graph has node names, but no direct node-type field and no `P31 -> Q5` edges. The original dataset does preserve Wikidata QIDs, so this notebook queries Wikidata for entities whose `instance of` is `human (Q5)` and caches the resulting integer node IDs in `R-GCN/checkpoints/wikidata_human_nodes.pkl`.

In [ ]:
if not NODE_ID_MAP.exists():
    raise FileNotFoundError(
        f"Missing {NODE_ID_MAP}. Upload ml_tkgl-smallpedia_nodeid.pkl so QIDs can be mapped to node IDs."
    )

with NODE_ID_MAP.open("rb") as f:
    qid_to_node = pickle.load(f)
node_to_qid = {node_id: qid for qid, node_id in qid_to_node.items()}


def fetch_human_qids(qids, batch_size=500, sleep_seconds=0.1):
    endpoint = "https://query.wikidata.org/sparql"
    human_qids = set()
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "290Final-VGAE-latent-demo/1.0 (Jupyter notebook)",
    }

    for start in range(0, len(qids), batch_size):
        batch = qids[start : start + batch_size]
        values = " ".join(f"wd:{qid}" for qid in batch)
        query = f"""
        SELECT ?item WHERE {{
          VALUES ?item {{ {values} }}
          ?item wdt:P31 wd:Q5 .
        }}
        """
        data = urllib.parse.urlencode({"query": query}).encode("utf-8")
        req = urllib.request.Request(endpoint, data=data, headers=headers, method="POST")
        with urllib.request.urlopen(req, timeout=60) as response:
            payload = json.loads(response.read().decode("utf-8"))
        for row in payload["results"]["bindings"]:
            human_qids.add(row["item"]["value"].rsplit("/", 1)[-1])
        if sleep_seconds:
            time.sleep(sleep_seconds)
        print(f"Checked {min(start + batch_size, len(qids)):,}/{len(qids):,}; humans so far: {len(human_qids):,}")
    return human_qids


if HUMAN_NODE_CACHE.exists():
    with HUMAN_NODE_CACHE.open("rb") as f:
        human_cache = pickle.load(f)
    human_qids = set(human_cache["human_qids"])
    human_node_ids = np.array(human_cache["human_node_ids"], dtype=np.int64)
    print(f"Loaded cached human node set from {HUMAN_NODE_CACHE}")
else:
    all_qids = sorted(qid_to_node)
    human_qids = fetch_human_qids(all_qids)
    human_node_ids = np.array(sorted(qid_to_node[qid] for qid in human_qids if qid in qid_to_node), dtype=np.int64)
    with HUMAN_NODE_CACHE.open("wb") as f:
        pickle.dump({"human_qids": sorted(human_qids), "human_node_ids": human_node_ids.tolist()}, f)
    print(f"Saved human node set to {HUMAN_NODE_CACHE}")

people_node_ids = human_node_ids
print(f"\nTrue Wikidata humans: {len(people_node_ids):,}")
print("Sample human labels:")
for node_id in people_node_ids[:30]:
    qid = node_to_qid.get(int(node_id), "?")
    print(f"  {node_id:5d} {qid:>10s}: {graph.node_labels.get(int(node_id), str(node_id))}")

In [ ]:
if not RGCN_CKPT.exists():
    raise FileNotFoundError(f"Missing {RGCN_CKPT}. Upload the trained R-GCN checkpoint.")
if not VGAE_RESULTS.exists():
    raise FileNotFoundError(
        f"Missing {VGAE_RESULTS}. Run R-GCN/wikidata_vgae_training.ipynb first."
    )

with VGAE_RESULTS.open("rb") as f:
    vgae_results = pickle.load(f)

vgae_config = vgae_results["config"]
hidden_dim = int(vgae_config.get("hidden_dim", 64))
latent_dim = int(vgae_config.get("latent_dim", hidden_dim))
feat_dim = graph.node_features.shape[1] if graph.node_features is not None else None

rgcn_model = RGCNLinkPredictor(
    num_nodes=graph.num_nodes,
    num_relations=graph.num_relations,
    hidden_dim=hidden_dim,
).to(DEVICE)
rgcn_model.load_state_dict(torch.load(RGCN_CKPT, map_location=DEVICE))
rgcn_model.eval()

vgae_model = VGAE(
    num_nodes=graph.num_nodes,
    hidden_dim=hidden_dim,
    num_relations=graph.num_relations,
    latent_dim=latent_dim,
    feat_dim=feat_dim,
).to(DEVICE)
vgae_model.load_state_dict(vgae_results["model_state_dict"])
vgae_model.eval()

print(f"Loaded R-GCN from {RGCN_CKPT}")
print(f"Loaded VGAE results from {VGAE_RESULTS}")
print(f"VGAE test metrics: {vgae_results.get('test_metrics')}")

In [ ]:
train_edges_device = graph.train_edges.to(DEVICE)
edge_index = train_edges_device[:, :2].t().contiguous()
edge_type = train_edges_device[:, 2]
node_features = graph.node_features.to(DEVICE) if graph.node_features is not None else None

with torch.no_grad():
    rgcn_latents = rgcn_model.encoder(
        edge_index,
        edge_type,
        node_features=node_features,
        num_nodes=graph.num_nodes,
    ).detach().cpu().numpy()
    _, vgae_mu, vgae_log_var = vgae_model(
        edge_index,
        edge_type,
        node_features=node_features,
        num_nodes=graph.num_nodes,
    )
    vgae_mu = vgae_mu.detach().cpu().numpy()
    vgae_std = torch.exp(0.5 * vgae_log_var).detach().cpu().numpy()

print(f"R-GCN latents: {rgcn_latents.shape}")
print(f"VGAE mu: {vgae_mu.shape}")
print(f"VGAE mean posterior std: {vgae_std.mean():.4f}")

In [ ]:
rng = np.random.default_rng(42)
max_points = 3000
plot_node_ids = people_node_ids
if len(plot_node_ids) > max_points:
    plot_node_ids = rng.choice(plot_node_ids, size=max_points, replace=False)

combined = np.vstack([rgcn_latents[plot_node_ids], vgae_mu[plot_node_ids]])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(combined)
rgcn_xy = coords[: len(plot_node_ids)]
vgae_xy = coords[len(plot_node_ids) :]

labels = [graph.node_labels.get(int(i), str(i)) for i in plot_node_ids]
rows = []
for model_name, xy in [("R-GCN", rgcn_xy), ("VGAE mu", vgae_xy)]:
    for node_id, label, point in zip(plot_node_ids, labels, xy):
        rows.append({"model": model_name, "node_id": int(node_id), "label": label, "x": point[0], "y": point[1]})

fig = px.scatter(
    rows,
    x="x",
    y="y",
    color="model",
    hover_name="label",
    hover_data={"node_id": True, "x": False, "y": False},
    opacity=0.65,
    title="Wikidata humans projected into a shared PCA view",
)
fig.update_traces(marker={"size": 5})
fig.show()

def normalized_nn_distances(x, sample_ids):
    subset = x[sample_ids]
    subset = (subset - subset.mean(axis=0, keepdims=True)) / (subset.std(axis=0, keepdims=True) + 1e-8)
    nn = NearestNeighbors(n_neighbors=2).fit(subset)
    distances, _ = nn.kneighbors(subset)
    return distances[:, 1]

nn_rows = []
for model_name, x in [("R-GCN", rgcn_latents), ("VGAE mu", vgae_mu)]:
    dists = normalized_nn_distances(x, plot_node_ids)
    nn_rows.extend({"model": model_name, "nearest_neighbor_distance": float(d)} for d in dists)
    print(f"{model_name} median normalized NN distance: {np.median(dists):.4f}")

fig = px.histogram(
    nn_rows,
    x="nearest_neighbor_distance",
    color="model",
    barmode="overlay",
    nbins=60,
    opacity=0.55,
    title="Local latent density: nearest-neighbor distances among Wikidata humans",
)
fig.show()

## Draggable Latent-Space Link Prediction

Drag the red probe point through the 2D PCA latent space. The demo maps that 2D location back through the PCA plane, scores candidate object nodes with the selected relation, and updates the top predictions in real time. Use the model dropdown to compare R-GCN and VGAE geometry.

In [ ]:
DEMO_RELATION_LABELS = [
    "occupation",
    "country of citizenship",
    "award received",
    "employer",
    "position held",
]

demo_rel_ids = [label_to_rel[label] for label in DEMO_RELATION_LABELS if label in label_to_rel]
demo_background_ids = plot_node_ids[:1500]


def make_model_payload(name, latents, decoder):
    local_pca = PCA(n_components=2, random_state=42)
    xy = local_pca.fit_transform(latents[demo_background_ids])
    relation_weights = decoder.relation_emb.weight.detach().cpu().numpy()
    payload = {
        "name": name,
        "points": [
            {
                "id": int(node_id),
                "x": round(float(point[0]), 5),
                "y": round(float(point[1]), 5),
                "label": graph.node_labels.get(int(node_id), str(node_id)),
            }
            for node_id, point in zip(demo_background_ids, xy)
        ],
        "mean": local_pca.mean_.astype(float).round(6).tolist(),
        "components": local_pca.components_.astype(float).round(6).tolist(),
        "relations": {},
    }
    for rel_id in demo_rel_ids:
        rel_edges = train_edges[train_edges[:, 2] == rel_id]
        object_ids, counts = torch.unique(rel_edges[:, 1], return_counts=True)
        top = torch.argsort(counts, descending=True)[:200]
        candidates = object_ids[top].cpu().numpy()
        payload["relations"][str(rel_id)] = {
            "label": rel_label(rel_id),
            "weight": relation_weights[rel_id].astype(float).round(6).tolist(),
            "candidates": [
                {
                    "id": int(node_id),
                    "label": graph.node_labels.get(int(node_id), str(node_id)),
                    "emb": latents[int(node_id)].astype(float).round(6).tolist(),
                }
                for node_id in candidates
            ],
        }
    return payload


demo_payload = {
    "models": {
        "rgcn": make_model_payload("R-GCN", rgcn_latents, rgcn_model.decoder),
        "vgae": make_model_payload("VGAE mu", vgae_mu, vgae_model.decoder),
    },
}

html_template = """
<div style='font-family: system-ui, sans-serif; max-width: 980px;'>
  <div style='display:flex; gap:16px; align-items:center; margin-bottom:10px;'>
    <label>Model <select id='modelSelect'></select></label>
    <label>Relation <select id='relSelect'></select></label>
    <span style='color:#555;'>Drag the red point.</span>
  </div>
  <div style='display:flex; gap:18px;'>
    <svg id='latentSvg' width='650' height='540' style='border:1px solid #bbb; background:#fafafa;'></svg>
    <div style='width:300px;'>
      <h3 style='margin-top:0;'>Nearest Human Sources</h3>
      <div id='coordText' style='font-size:12px; color:#555; margin-bottom:8px;'></div>
      <ol id='nearestList' style='padding-left:22px; margin-bottom:22px;'></ol>
      <h3>Top Link Predictions</h3>
      <ol id='predList' style='padding-left:22px;'></ol>
    </div>
  </div>
</div>
<script>
const payload = __DEMO_PAYLOAD__;
const svg = document.getElementById('latentSvg');
const modelSelect = document.getElementById('modelSelect');
const relSelect = document.getElementById('relSelect');
const predList = document.getElementById('predList');
const nearestList = document.getElementById('nearestList');
const coordText = document.getElementById('coordText');
const W = 650, H = 540, pad = 34;
let probe = {x: 0, y: 0};
let dragging = false;

for (const [key, model] of Object.entries(payload.models)) {
  const opt = document.createElement('option'); opt.value = key; opt.textContent = model.name; modelSelect.appendChild(opt);
}

function currentModel() { return payload.models[modelSelect.value]; }
function currentRelation() { return currentModel().relations[relSelect.value]; }
function extents(points) {
  const xs = points.map(p => p.x), ys = points.map(p => p.y);
  const xpad = (Math.max(...xs) - Math.min(...xs)) * 0.08 || 1;
  const ypad = (Math.max(...ys) - Math.min(...ys)) * 0.08 || 1;
  return {xmin: Math.min(...xs)-xpad, xmax: Math.max(...xs)+xpad, ymin: Math.min(...ys)-ypad, ymax: Math.max(...ys)+ypad};
}
let scale = extents(currentModel().points);
function sx(x) { return pad + (x-scale.xmin)/(scale.xmax-scale.xmin)*(W-2*pad); }
function sy(y) { return H-pad - (y-scale.ymin)/(scale.ymax-scale.ymin)*(H-2*pad); }
function dx(px) { return scale.xmin + (px-pad)/(W-2*pad)*(scale.xmax-scale.xmin); }
function dy(py) { return scale.ymin + (H-pad-py)/(H-2*pad)*(scale.ymax-scale.ymin); }

function fillRelations() {
  relSelect.innerHTML = '';
  for (const [relId, rel] of Object.entries(currentModel().relations)) {
    const opt = document.createElement('option'); opt.value = relId; opt.textContent = rel.label; relSelect.appendChild(opt);
  }
}

function latentFromProbe() {
  const m = currentModel();
  const z = m.mean.slice();
  for (let j = 0; j < z.length; j++) z[j] += probe.x*m.components[0][j] + probe.y*m.components[1][j];
  return z;
}
function score(z, relWeight, emb) {
  let s = 0; for (let i = 0; i < z.length; i++) s += z[i] * relWeight[i] * emb[i]; return s;
}
function updateScores() {
  const rel = currentRelation();
  const z = latentFromProbe();
  const nearest = currentModel().points.map(p => ({...p, dist: Math.hypot(p.x - probe.x, p.y - probe.y)}));
  nearest.sort((a,b) => a.dist - b.dist);
  nearestList.innerHTML = '';
  for (const p of nearest.slice(0, 8)) {
    const li = document.createElement('li');
    li.textContent = `${p.label}  (${p.dist.toFixed(2)})`;
    nearestList.appendChild(li);
  }
  const scored = rel.candidates.map(c => ({...c, score: score(z, rel.weight, c.emb)}));
  scored.sort((a,b) => b.score - a.score);
  predList.innerHTML = '';
  for (const c of scored.slice(0, 12)) {
    const li = document.createElement('li');
    li.textContent = `${c.label}  (${c.score.toFixed(3)})`;
    predList.appendChild(li);
  }
  coordText.textContent = `PCA point: (${probe.x.toFixed(2)}, ${probe.y.toFixed(2)})`;
}
function draw() {
  const model = currentModel();
  scale = extents(model.points.concat([probe]));
  svg.innerHTML = '';
  const axis = document.createElementNS('http://www.w3.org/2000/svg', 'rect');
  axis.setAttribute('x', pad); axis.setAttribute('y', pad); axis.setAttribute('width', W-2*pad); axis.setAttribute('height', H-2*pad);
  axis.setAttribute('fill', 'white'); axis.setAttribute('stroke', '#ddd'); svg.appendChild(axis);
  for (const p of model.points) {
    const c = document.createElementNS('http://www.w3.org/2000/svg', 'circle');
    c.setAttribute('cx', sx(p.x)); c.setAttribute('cy', sy(p.y)); c.setAttribute('r', 2.4);
    c.setAttribute('fill', '#4b7bec'); c.setAttribute('opacity', 0.42);
    const title = document.createElementNS('http://www.w3.org/2000/svg', 'title'); title.textContent = p.label; c.appendChild(title);
    svg.appendChild(c);
  }
  const probeCircle = document.createElementNS('http://www.w3.org/2000/svg', 'circle');
  probeCircle.setAttribute('id', 'probe'); probeCircle.setAttribute('cx', sx(probe.x)); probeCircle.setAttribute('cy', sy(probe.y)); probeCircle.setAttribute('r', 9);
  probeCircle.setAttribute('fill', '#e74c3c'); probeCircle.setAttribute('stroke', '#8e1f17'); probeCircle.setAttribute('stroke-width', 2); probeCircle.style.cursor = 'grab';
  svg.appendChild(probeCircle);
  updateScores();
}
svg.addEventListener('mousedown', e => { if (e.target.id === 'probe') dragging = true; });
window.addEventListener('mouseup', () => { dragging = false; });
svg.addEventListener('mousemove', e => {
  if (!dragging) return;
  const r = svg.getBoundingClientRect();
  const px = Math.max(pad, Math.min(W-pad, e.clientX - r.left));
  const py = Math.max(pad, Math.min(H-pad, e.clientY - r.top));
  probe.x = dx(px); probe.y = dy(py);
  document.getElementById('probe').setAttribute('cx', sx(probe.x));
  document.getElementById('probe').setAttribute('cy', sy(probe.y));
  updateScores();
});
modelSelect.addEventListener('change', () => { probe = {x:0, y:0}; fillRelations(); draw(); });
relSelect.addEventListener('change', updateScores);
fillRelations(); draw();
</script>
"""
html = html_template.replace("__DEMO_PAYLOAD__", json.dumps(demo_payload))

display(HTML(html))